# Document Question Answering System (RAG)

**Author:** Arpita Das
**Assignment:** Week 7 — Build a simple Retrieval-Augmented Generation (RAG) pipeline that answers
questions from a custom document collection.

### What this notebook builds
1. **Ingestion** — load `.txt` / `.pdf` files (or paste raw text) into the pipeline.
2. **Chunking** — split long documents into overlapping, retrieval-friendly chunks.
3. **Embedding** — turn chunks (and queries) into dense vectors with a pre-trained sentence embedding model.
4. **Vector store** — index the chunk embeddings in FAISS for fast similarity search.
5. **Retrieval** — fetch the top-k most relevant chunks for a user question.
6. **Hybrid search (optional)** — blend BM25 keyword search with vector search.
7. **Re-ranking (optional)** — re-score retrieved chunks with a cross-encoder for precision.
8. **Generation** — stuff the retrieved context + question into a prompt and generate a grounded answer with a language model.
9. **Evaluation** — run sample questions end-to-end and print a system metrics report.

> Replace the demo document in **Section 1** with your own PDF/notes/resume/research paper — the rest
> of the pipeline works unchanged.


## 0. Setup — install dependencies
Run once per Colab session.

In [ ]:
!pip install -q rank_bm25==0.2.2 pypdf==4.3.1 sentence-transformers faiss-cpu transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.8/295.8 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 44.2 MB/s eta 0:00:00


In [ ]:
import os
import re
import textwrap
import time
import numpy as np
import pandas as pd

from pypdf import PdfReader
from sentence_transformers import SentenceTransformer, CrossEncoder
import faiss
from rank_bm25 import BM25Okapi
from transformers import pipeline

pd.set_option("display.max_colwidth", 120)
np.random.seed(42)


## 1. Document Ingestion

The ingestion module accepts:
- plain `.txt` files
- `.pdf` files (parsed page-by-page with `pypdf`)
- raw pasted text (for quick demos, or a Hugging Face text dataset dropped into a string)

For this notebook we generate a small **demo knowledge base** (an original write-up on machine
learning basics) so the pipeline runs end-to-end without any external files. Swap in your own
document(s) by uploading them and pointing `DOC_PATHS` at them.


In [ ]:
def load_txt(path: str) -> str:
    with open(path, "r", encoding="utf-8") as f:
        return f.read()


def load_pdf(path: str) -> str:
    reader = PdfReader(path)
    pages = [page.extract_text() or "" for page in reader.pages]
    return "\n".join(pages)


def load_document(path: str) -> str:
    """Dispatch to the right loader based on file extension."""
    ext = os.path.splitext(path)[1].lower()
    if ext == ".pdf":
        return load_pdf(path)
    elif ext in (".txt", ".md"):
        return load_txt(path)
    else:
        raise ValueError(f"Unsupported file type: {ext}")


In [ ]:
os.makedirs("data", exist_ok=True)
with open("data/ml_notes.txt", "w", encoding="utf-8") as f:
    f.write("""MACHINE LEARNING: A BEGINNER'S NOTES

Chapter 1: What is Machine Learning?
Machine learning is a branch of artificial intelligence in which a computer program improves its
performance on a task by learning patterns from data instead of following hand written rules. Rather
than a programmer explicitly coding every decision, a model is exposed to examples and adjusts its
internal parameters so that its predictions get closer to the correct answers over time.

Chapter 2: Supervised Learning
In supervised learning every training example comes with a known label or target value. The two
common families of supervised tasks are classification, where the target is a category such as
"spam" or "not spam", and regression, where the target is a continuous number such as a house price.
Popular supervised algorithms include linear regression, logistic regression, decision trees, random
forests, and gradient boosted trees.

Chapter 3: Unsupervised Learning
Unsupervised learning works with data that has no labels. The goal is usually to discover structure
in the data on its own. Clustering algorithms such as K-Means group similar records together, while
dimensionality reduction techniques such as PCA compress many correlated features into a smaller set
of components that still capture most of the variation in the data.

Chapter 4: Overfitting and Regularization
A model overfits when it memorizes the training data, including its noise, and therefore performs
poorly on new, unseen data. Common ways to fight overfitting include gathering more training data,
simplifying the model, using regularization terms such as L1 or L2 penalties, and using techniques
like dropout in neural networks. Cross-validation is the standard way to estimate how well a model
will generalize before it is deployed.

Chapter 5: Neural Networks
A neural network is built from layers of simple units, often called neurons, that each compute a
weighted sum of their inputs followed by a non-linear activation function. Stacking many layers lets
the network learn increasingly abstract representations of the input. Convolutional neural networks
are well suited to images because they exploit local spatial structure, while recurrent and
transformer architectures are well suited to sequential data such as text.

Chapter 6: Evaluation Metrics
Choosing the right metric matters. For classification, accuracy can be misleading when classes are
imbalanced, so precision, recall, F1-score, and ROC-AUC are often reported alongside it. For
regression, mean absolute error, mean squared error, and R-squared are the usual choices. The metric
should reflect what actually matters for the business problem being solved.

Chapter 7: Retrieval-Augmented Generation
Retrieval-Augmented Generation, or RAG, combines a retrieval system with a language model. Instead of
relying only on what the language model memorized during training, the system first searches a
document collection for the passages most relevant to a question, then passes those passages to the
language model as context. This lets the model answer questions about private or up-to-date documents
it has never seen before, and it makes answers easier to verify because the supporting text is known.
""")

# Point this list at your own files to use your own documents instead
DOC_PATHS = ["data/ml_notes.txt"]

raw_documents = {path: load_document(path) for path in DOC_PATHS}
for path, text in raw_documents.items():
    print(f"{path}: {len(text)} characters, {len(text.split())} words")


data/ml_notes.txt: 3240 characters, 487 words


## 2. Chunking

Language-model context windows and embedding models both work best on short, focused passages, so
each document is split into overlapping word-level chunks. The overlap keeps a sentence that was
cut at a chunk boundary readable in at least one of the two chunks it touches.


In [ ]:
def chunk_text(text: str, chunk_size: int = 120, overlap_sentences: int = 2):
    """Split text into chunks of roughly `chunk_size` words, built up from WHOLE sentences so a
    chunk never starts or ends mid-sentence. `overlap_sentences` trailing sentences from one
    chunk are carried into the next, so a sentence near a boundary stays readable in both."""
    sentences = re.split(r"(?<=[.!?])\s+", text.strip())
    sentences = [s.strip() for s in sentences if s.strip()]

    chunks = []
    current, current_words = [], 0
    for sent in sentences:
        sent_len = len(sent.split())
        if current_words + sent_len > chunk_size and current:
            chunks.append(" ".join(current))
            current = current[-overlap_sentences:]
            current_words = sum(len(s.split()) for s in current)
        current.append(sent)
        current_words += sent_len
    if current:
        chunks.append(" ".join(current))
    return chunks


CHUNK_SIZE = 120
CHUNK_OVERLAP = 2  # now measured in overlapping SENTENCES, not words

all_chunks = []          # chunk text
chunk_sources = []       # which document each chunk came from

for path, text in raw_documents.items():
    doc_chunks = chunk_text(text, CHUNK_SIZE, CHUNK_OVERLAP)
    all_chunks.extend(doc_chunks)
    chunk_sources.extend([path] * len(doc_chunks))

print(f"Total chunks: {len(all_chunks)}")
print(f"Average chunk length: {np.mean([len(c.split()) for c in all_chunks]):.1f} words\n")
print("Sample chunk:\n" + textwrap.fill(all_chunks[0], 100))


Total chunks: 7
Average chunk length: 106.1 words

Sample chunk:
MACHINE LEARNING: A BEGINNER'S NOTES  Chapter 1: What is Machine Learning? Machine learning is a
branch of artificial intelligence in which a computer program improves its performance on a task by
learning patterns from data instead of following hand written rules. Rather than a programmer
explicitly coding every decision, a model is exposed to examples and adjusts its internal parameters
so that its predictions get closer to the correct answers over time. Chapter 2: Supervised Learning
In supervised learning every training example comes with a known label or target value.


## 3. Embeddings

`all-MiniLM-L6-v2` is a small, fast sentence-transformer that maps text to a 384-dimensional dense
vector. Semantically similar text ends up close together in this vector space, which is what makes
similarity search work.


In [ ]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

chunk_embeddings = embedding_model.encode(
    all_chunks, show_progress_bar=True, convert_to_numpy=True, normalize_embeddings=True
)

print(f"Embedding matrix shape: {chunk_embeddings.shape}")
EMBED_DIM = chunk_embeddings.shape[1]


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embedding matrix shape: (7, 384)


## 3.1 Chunk-Size Experiment

`CHUNK_SIZE = 120` (with 20-word overlap) was picked as a default, not proven — this section checks
that choice instead of assuming it. Three chunk sizes are built from the same source document and
compared on (a) how many chunks they produce and (b) whether the single most relevant chunk changes
for a fixed test query. This is the "adjust chunk borders" optimization called out in the brief.


In [ ]:
# Build alternate chunkings of the same source document(s) to compare against the
# CHUNK_SIZE/CHUNK_OVERLAP chosen above. This does NOT change all_chunks / chunk_sources —
# it is a side-by-side comparison only.
chunk_size_options = [80, 120, 200]
chunk_experiment_rows = []
chunk_variants = {}

for size in chunk_size_options:
    overlap = 2  # fixed sentence-overlap across variants, so only chunk_size varies
    variant_chunks = []
    variant_sources = []
    for path, text in raw_documents.items():
        c = chunk_text(text, size, overlap)
        variant_chunks.extend(c)
        variant_sources.extend([path] * len(c))
    chunk_variants[size] = {"chunks": variant_chunks, "sources": variant_sources}
    chunk_experiment_rows.append({
        "chunk_size": size,
        "overlap": overlap,
        "num_chunks": len(variant_chunks),
        "avg_words_per_chunk": round(np.mean([len(c.split()) for c in variant_chunks]), 1),
    })

pd.DataFrame(chunk_experiment_rows)


,chunk_size,overlap,num_chunks,avg_words_per_chunk
0,80,2,16,74.1
1,120,2,7,106.1
2,200,2,4,152.8


In [ ]:
# For a fixed query, embed each chunk-size variant separately and check which top-1
# chunk it retrieves. If the "winning" passage stays essentially the same across sizes,
# the pipeline is not overly sensitive to this hyperparameter; if it swings wildly, that is
# worth flagging in the notes.
probe_query = "What causes overfitting and how can it be prevented?"

for size, variant in chunk_variants.items():
    v_embeddings = embedding_model.encode(
        variant["chunks"], convert_to_numpy=True, normalize_embeddings=True
    )
    v_index = faiss.IndexFlatIP(v_embeddings.shape[1])
    v_index.add(v_embeddings)
    q_vec = embedding_model.encode([probe_query], convert_to_numpy=True, normalize_embeddings=True)
    score, idx = v_index.search(q_vec, 1)
    top_chunk = variant["chunks"][idx[0][0]]
    print(f"chunk_size={size:>3} | top score={score[0][0]:.3f}")
    print(textwrap.fill(top_chunk, 100)[:220], "...\n")


chunk_size= 80 | top score=0.677
Chapter 4: Overfitting and Regularization A model overfits when it memorizes the training data,
including its noise, and therefore performs poorly on new, unseen data. Common ways to fight
overfitting include gathering m ...

chunk_size=120 | top score=0.510
Common ways to fight overfitting include gathering more training data, simplifying the model, using
regularization terms such as L1 or L2 penalties, and using techniques like dropout in neural
networks. Cross-validation  ...

chunk_size=200 | top score=0.464
Chapter 3: Unsupervised Learning Unsupervised learning works with data that has no labels. The goal
is usually to discover structure in the data on its own. Clustering algorithms such as K-Means group
similar records tog ...



## 4. Vector Store

FAISS (`IndexFlatIP`) stores the normalized chunk embeddings and supports fast cosine-similarity
search via inner product. For larger corpora an approximate index (e.g. `IndexIVFFlat`) would trade
a little accuracy for much faster lookups.


In [ ]:
index = faiss.IndexFlatIP(EMBED_DIM)
index.add(chunk_embeddings)
print(f"Vectors stored in index: {index.ntotal}")


Vectors stored in index: 7


## 5. Retrieval

A user's question is embedded with the same model, and the vector store returns the chunks whose
embeddings are closest to the query embedding.


In [ ]:
def vector_search(query: str, top_k: int = 3):
    query_vec = embedding_model.encode([query], convert_to_numpy=True, normalize_embeddings=True)
    scores, idxs = index.search(query_vec, top_k)
    results = []
    for score, idx in zip(scores[0], idxs[0]):
        results.append({"chunk": all_chunks[idx], "source": chunk_sources[idx], "score": float(score)})
    return results


for r in vector_search("What is overfitting and how do you prevent it?", top_k=2):
    print(f"[score={r['score']:.3f}] {r['source']}")
    print(textwrap.fill(r["chunk"], 100), "\n")


[score=0.528] data/ml_notes.txt
Common ways to fight overfitting include gathering more training data, simplifying the model, using
regularization terms such as L1 or L2 penalties, and using techniques like dropout in neural
networks. Cross-validation is the standard way to estimate how well a model will generalize before
it is deployed. Chapter 5: Neural Networks A neural network is built from layers of simple units,
often called neurons, that each compute a weighted sum of their inputs followed by a non-linear
activation function. Stacking many layers lets the network learn increasingly abstract
representations of the input. Convolutional neural networks are well suited to images because they
exploit local spatial structure, while recurrent and transformer architectures are well suited to
sequential data such as text. 

[score=0.521] data/ml_notes.txt
Clustering algorithms such as K-Means group similar records together, while dimensionality reduction
techniques such as PCA compress m

## 6. Hybrid Search — keyword (BM25) + vector

Pure vector search can miss exact keyword matches (e.g. acronyms or names). Hybrid search blends
a BM25 keyword score with the vector similarity score so both exact terms and paraphrased questions
are handled well.


In [ ]:
tokenized_chunks = [c.lower().split() for c in all_chunks]
bm25 = BM25Okapi(tokenized_chunks)


def _normalize(scores: np.ndarray) -> np.ndarray:
    if scores.max() == scores.min():
        return np.zeros_like(scores)
    return (scores - scores.min()) / (scores.max() - scores.min())


def hybrid_search(query: str, top_k: int = 3, alpha: float = 0.5):
    """alpha weights vector similarity; (1 - alpha) weights BM25 keyword score."""
    query_vec = embedding_model.encode([query], convert_to_numpy=True, normalize_embeddings=True)
    vector_scores, order = index.search(query_vec, len(all_chunks))
    vector_scores, order = vector_scores[0], order[0]
    vec_full = np.zeros(len(all_chunks))
    vec_full[order] = vector_scores

    bm25_scores = np.array(bm25.get_scores(query.lower().split()))

    blended = alpha * _normalize(vec_full) + (1 - alpha) * _normalize(bm25_scores)
    top_idx = np.argsort(blended)[::-1][:top_k]
    return [
        {"chunk": all_chunks[i], "source": chunk_sources[i], "score": float(blended[i])}
        for i in top_idx
    ]


for r in hybrid_search("RAG retrieval augmented generation", top_k=2):
    print(f"[blended score={r['score']:.3f}] {r['source']}")
    print(textwrap.fill(r["chunk"], 100), "\n")


[blended score=1.000] data/ml_notes.txt
The metric should reflect what actually matters for the business problem being solved. Chapter 7:
Retrieval-Augmented Generation Retrieval-Augmented Generation, or RAG, combines a retrieval system
with a language model. Instead of relying only on what the language model memorized during training,
the system first searches a document collection for the passages most relevant to a question, then
passes those passages to the language model as context. This lets the model answer questions about
private or up-to-date documents it has never seen before, and it makes answers easier to verify
because the supporting text is known. 

[blended score=0.920] data/ml_notes.txt
Stacking many layers lets the network learn increasingly abstract representations of the input.
Convolutional neural networks are well suited to images because they exploit local spatial
structure, while recurrent and transformer architectures are well suited to sequential data such as
t

## 7. Re-ranking (optional)

A cross-encoder scores the (query, chunk) pair jointly rather than comparing two independent
embeddings, which is slower but more accurate. It is used here only to re-order a small candidate
set returned by the cheaper first-stage retrieval — that keeps it fast enough for interactive use.


In [ ]:
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")


def rerank(query: str, candidates: list, top_k: int = 3):
    pairs = [(query, c["chunk"]) for c in candidates]
    scores = reranker.predict(pairs)
    for c, s in zip(candidates, scores):
        c["rerank_score"] = float(s)
    return sorted(candidates, key=lambda c: c["rerank_score"], reverse=True)[:top_k]


candidates = vector_search("How do neural networks process images?", top_k=5)
reranked = rerank("How do neural networks process images?", candidates, top_k=2)
for r in reranked:
    print(f"[rerank_score={r['rerank_score']:.3f}] {r['source']}")
    print(textwrap.fill(r["chunk"], 100), "\n")


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

[rerank_score=0.811] data/ml_notes.txt
Common ways to fight overfitting include gathering more training data, simplifying the model, using
regularization terms such as L1 or L2 penalties, and using techniques like dropout in neural
networks. Cross-validation is the standard way to estimate how well a model will generalize before
it is deployed. Chapter 5: Neural Networks A neural network is built from layers of simple units,
often called neurons, that each compute a weighted sum of their inputs followed by a non-linear
activation function. Stacking many layers lets the network learn increasingly abstract
representations of the input. Convolutional neural networks are well suited to images because they
exploit local spatial structure, while recurrent and transformer architectures are well suited to
sequential data such as text. 

[rerank_score=0.223] data/ml_notes.txt
Stacking many layers lets the network learn increasingly abstract representations of the input.
Convolutional neural net

## 7.1 Retrieval Evaluation

The sections above build vector, hybrid, and re-ranked retrieval but never check whether any of them
actually retrieve the *right* passage. Here a small labeled evaluation set — each question paired
with a keyword that should appear in a correct retrieval — is used to compute a **hit-rate@k** for
each retrieval method, so the choice of method is backed by a number instead of intuition.


In [ ]:
# Each question is paired with a keyword/phrase that should show up in a genuinely
# relevant chunk. This avoids hard-coding exact chunk indices (which would break the moment
# someone swaps in their own document) while still giving an objective correctness check.
eval_set = [
    ("What is supervised learning?", "supervised"),
    ("How does K-Means clustering work?", "k-means"),
    ("What is overfitting?", "overfit"),
    ("What does RAG stand for?", "retrieval-augmented generation"),
    ("What is a convolutional neural network used for?", "convolutional"),
]


def hit_rate(search_fn, eval_set, top_k=3):
    hits = 0
    for question, keyword in eval_set:
        results = search_fn(question, top_k=top_k)
        found = any(keyword.lower() in r["chunk"].lower() for r in results)
        hits += int(found)
    return hits / len(eval_set)


def vector_then_rerank(question, top_k=3):
    return rerank(question, vector_search(question, top_k=top_k * 2), top_k=top_k)


retrieval_eval = pd.DataFrame([
    {"method": "vector only", "hit_rate@3": hit_rate(vector_search, eval_set)},
    {"method": "hybrid (BM25 + vector)", "hit_rate@3": hit_rate(hybrid_search, eval_set)},
    {"method": "vector + rerank", "hit_rate@3": hit_rate(vector_then_rerank, eval_set)},
])
retrieval_eval


,method,hit_rate@3
0,vector only,1.0
1,hybrid (BM25 + vector),1.0
2,vector + rerank,1.0


## 7.2 Method Comparison — Same Question, Three Retrievers

Rather than trusting each method in isolation, this runs the same questions through vector-only,
hybrid, and reranked retrieval side by side and shows the top-1 chunk each one returns, so any
disagreement between methods is visible directly instead of buried in separate cells above.


In [ ]:
comparison_rows = []
for question, _ in eval_set:
    vec_top = vector_search(question, top_k=1)[0]
    hyb_top = hybrid_search(question, top_k=1)[0]
    rer_top = vector_then_rerank(question, top_k=1)[0]
    comparison_rows.append({
        "question": question,
        "vector_top1": vec_top["chunk"][:90] + "...",
        "hybrid_top1": hyb_top["chunk"][:90] + "...",
        "reranked_top1": rer_top["chunk"][:90] + "...",
    })

comparison_df = pd.DataFrame(comparison_rows)
comparison_df


,question,vector_top1,hybrid_top1,reranked_top1
0,What is supervised learning?,MACHINE LEARNING: A BEGINNER'S NOTES\n\nChapter 1: What is Machine Learning? Machine learnin...,MACHINE LEARNING: A BEGINNER'S NOTES\n\nChapter 1: What is Machine Learning? Machine learnin...,MACHINE LEARNING: A BEGINNER'S NOTES\n\nChapter 1: What is Machine Learning? Machine learnin...
1,How does K-Means clustering work?,"Clustering algorithms such as K-Means group similar records together, while\ndimensionality...","Clustering algorithms such as K-Means group similar records together, while\ndimensionality...","Clustering algorithms such as K-Means group similar records together, while\ndimensionality..."
2,What is overfitting?,"Common ways to fight overfitting include gathering more training data,\nsimplifying the mod...",MACHINE LEARNING: A BEGINNER'S NOTES\n\nChapter 1: What is Machine Learning? Machine learnin...,"Clustering algorithms such as K-Means group similar records together, while\ndimensionality..."
3,What does RAG stand for?,The metric\nshould reflect what actually matters for the business problem being solved. Cha...,The metric\nshould reflect what actually matters for the business problem being solved. Cha...,The metric\nshould reflect what actually matters for the business problem being solved. Cha...
4,What is a convolutional neural network used for?,"Common ways to fight overfitting include gathering more training data,\nsimplifying the mod...","Common ways to fight overfitting include gathering more training data,\nsimplifying the mod...",Stacking many layers lets\nthe network learn increasingly abstract representations of the i...


## 7.3 Embedding Model Comparison

The pipeline so far only tries one embedding model (`all-MiniLM-L6-v2`). This builds a second index
with a different sentence-transformer and compares retrieval `hit_rate@3` against the same
`eval_set` used above, so the embedding-model choice is backed by a number rather than assumed to
be fine.


In [ ]:
# A second, smaller/faster embedding model for comparison. Both models produce 384-dim
# vectors here, so the comparison is about retrieval QUALITY, not just speed or size.
alt_embedding_model = SentenceTransformer("paraphrase-MiniLM-L3-v2")

alt_embeddings = alt_embedding_model.encode(
    all_chunks, convert_to_numpy=True, normalize_embeddings=True
)
alt_index = faiss.IndexFlatIP(alt_embeddings.shape[1])
alt_index.add(alt_embeddings)


def alt_vector_search(query: str, top_k: int = 3):
    q_vec = alt_embedding_model.encode([query], convert_to_numpy=True, normalize_embeddings=True)
    scores, idxs = alt_index.search(q_vec, top_k)
    return [
        {"chunk": all_chunks[i], "source": chunk_sources[i], "score": float(s)}
        for s, i in zip(scores[0], idxs[0])
    ]


embedding_comparison = pd.DataFrame([
    {
        "embedding_model": "all-MiniLM-L6-v2",
        "dimension": EMBED_DIM,
        "hit_rate@3": hit_rate(vector_search, eval_set),
    },
    {
        "embedding_model": "paraphrase-MiniLM-L3-v2",
        "dimension": alt_embeddings.shape[1],
        "hit_rate@3": hit_rate(alt_vector_search, eval_set),
    },
])
embedding_comparison


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.83k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

,embedding_model,dimension,hit_rate@3
0,all-MiniLM-L6-v2,384,1.0
1,paraphrase-MiniLM-L3-v2,384,1.0


## 8. Prompt Construction & Generation

The retrieved chunks are concatenated into a context block and combined with the question in an
instruction prompt, then passed to a text-generation model. `google/flan-t5-base` is small enough to
run on a Colab CPU/GPU and follows instruction-style prompts well for this kind of grounded QA.


In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

gen_tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
gen_model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")


def build_prompt(question: str, contexts: list) -> str:
    context_block = "\n\n".join(f"[{i+1}] {c['chunk']}" for i, c in enumerate(contexts))
    return (
        "Answer the question using ONLY the context below. "
        "If the answer is not contained in the context, say you don't know.\n\n"
        f"Context:\n{context_block}\n\n"
        f"Question: {question}\n"
        "Answer:"
    )


def generate_answer(question: str, contexts: list) -> str:
    prompt = build_prompt(question, contexts)
    inputs = gen_tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
    output_ids = gen_model.generate(**inputs, max_new_tokens=200)
    return gen_tokenizer.decode(output_ids[0], skip_special_tokens=True).strip()

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  990MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

## 8.1 Language Model Comparison

Compares the default generation model (`flan-t5-base`) against a smaller variant
(`flan-t5-small`) on the same retrieved context for each `eval_set` question, checking both
whether each answer contains the expected keyword and how the two models differ in practice —
the "experiment with different language models" item from the brief.


In [ ]:
alt_gen_tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")
alt_gen_model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")


def generate_answer_alt(question: str, contexts: list) -> str:
    prompt = build_prompt(question, contexts)
    inputs = alt_gen_tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
    output_ids = alt_gen_model.generate(**inputs, max_new_tokens=200)
    return alt_gen_tokenizer.decode(output_ids[0], skip_special_tokens=True).strip()


llm_comparison_rows = []
for question, expected_keyword in eval_set:
    contexts = vector_then_rerank(question, top_k=3)
    base_answer = generate_answer(question, contexts)
    alt_answer = generate_answer_alt(question, contexts)
    llm_comparison_rows.append({
        "question": question,
        "flan-t5-base_answer": base_answer,
        "flan-t5-base_pass": expected_keyword.lower() in base_answer.lower(),
        "flan-t5-small_answer": alt_answer,
        "flan-t5-small_pass": expected_keyword.lower() in alt_answer.lower(),
    })

llm_comparison_df = pd.DataFrame(llm_comparison_rows)
print(f"flan-t5-base accuracy:  {llm_comparison_df['flan-t5-base_pass'].mean():.0%}")
print(f"flan-t5-small accuracy: {llm_comparison_df['flan-t5-small_pass'].mean():.0%}")
llm_comparison_df


config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

flan-t5-base accuracy:  20%
flan-t5-small accuracy: 20%


,question,flan-t5-base_answer,flan-t5-base_pass,flan-t5-small_answer,flan-t5-small_pass
0,What is supervised learning?,a computer program improves its performance on a task by learning patterns from data instead of following hand writt...,False,Machine learning is a branch of artificial intelligence,False
1,How does K-Means clustering work?,[1],False,[1],False
2,What is overfitting?,noise,False,[2],False
3,What does RAG stand for?,Retrieval-Augmented Generation,True,Retrieval-Augmented Generation,True
4,What is a convolutional neural network used for?,Chapter 5: Neural Networks,False,[2],False


## 9. End-to-End RAG Pipeline

`rag_answer` ties retrieval and generation together: hybrid search finds candidates, the
cross-encoder re-ranks them, and the top passages are handed to the language model.


In [ ]:
def rag_answer(question: str, top_k: int = 3, use_hybrid: bool = True, use_reranker: bool = True):
    candidates = hybrid_search(question, top_k=top_k * 2) if use_hybrid else vector_search(question, top_k=top_k * 2)
    contexts = rerank(question, candidates, top_k=top_k) if use_reranker else candidates[:top_k]
    answer = generate_answer(question, contexts)
    return {"question": question, "answer": answer, "contexts": contexts}


## 10. Testing & Validation

Runs the full pipeline on the labeled `eval_set` from Section 7.1 and checks each generated answer
against its expected keyword, so this section reports a pass/fail and an overall accuracy score
instead of just printing answers for a human to eyeball.


In [ ]:
# Reuse the labeled eval_set from Section 7.1 (question + expected keyword) so "validation"
# here means something concrete: did the generated ANSWER actually contain the expected term,
# not just "did the pipeline run without crashing".
validation_log = []
for question, expected_keyword in eval_set:
    result = rag_answer(question)
    passed = expected_keyword.lower() in result["answer"].lower()
    validation_log.append({
        "question": question,
        "answer": result["answer"],
        "expected_keyword": expected_keyword,
        "passed": passed,
        "top_source": result["contexts"][0]["source"],
        "top_score": round(result["contexts"][0].get("rerank_score", result["contexts"][0]["score"]), 3),
    })
    print(f"Q: {question}")
    print(f"A: {result['answer']}")
    status = "PASS" if passed else "FAIL"
    print(f"{status} (expected keyword: '{expected_keyword}')\n")

validation_df = pd.DataFrame(validation_log)
accuracy = validation_df["passed"].mean()
num_passed = validation_df["passed"].sum()
print(f"Answer accuracy: {accuracy:.0%} ({num_passed}/{len(validation_df)})")
validation_df


Q: What is supervised learning?
A: a computer program improves its performance on a task by learning patterns from data instead of following hand written rules
FAIL (expected keyword: 'supervised')

Q: How does K-Means clustering work?
A: [1]
FAIL (expected keyword: 'k-means')

Q: What is overfitting?
A: noise
FAIL (expected keyword: 'overfit')

Q: What does RAG stand for?
A: Retrieval-Augmented Generation
PASS (expected keyword: 'retrieval-augmented generation')

Q: What is a convolutional neural network used for?
A: Chapter 5: Neural Networks
FAIL (expected keyword: 'convolutional')

Answer accuracy: 20% (1/5)


,question,answer,expected_keyword,passed,top_source,top_score
0,What is supervised learning?,a computer program improves its performance on a task by learning patterns from data instead of following hand writt...,supervised,False,data/ml_notes.txt,5.861
1,How does K-Means clustering work?,[1],k-means,False,data/ml_notes.txt,4.782
2,What is overfitting?,noise,overfit,False,data/ml_notes.txt,3.792
3,What does RAG stand for?,Retrieval-Augmented Generation,retrieval-augmented generation,True,data/ml_notes.txt,1.965
4,What is a convolutional neural network used for?,Chapter 5: Neural Networks,convolutional,False,data/ml_notes.txt,3.068


## 10.1 Edge Case — Out-of-Scope Question

A robust RAG system should say it doesn't know rather than hallucinate when the answer genuinely
isn't in the document. This deliberately asks something the source document never covers.


In [ ]:
out_of_scope_question = "What is the capital of France?"
edge_result = rag_answer(out_of_scope_question)
print(f"Q: {out_of_scope_question}")
print(f"A: {edge_result['answer']}")
print(f"Top retrieved score: {edge_result['contexts'][0].get('rerank_score', edge_result['contexts'][0]['score']):.3f}")
# A low top score alongside an "I don't know"-style answer indicates the grounding instruction
# in the prompt is doing its job instead of the model inventing an answer.


Q: What is the capital of France?
A: not known
Top retrieved score: -11.091


## 10.2 Latency

Reports how much of the end-to-end answer time is retrieval versus generation, since generation
(loading + running the language model) is typically the dominant cost.


In [ ]:
latency_question = "What is the difference between supervised and unsupervised learning?"

t0 = time.perf_counter()
_candidates = hybrid_search(latency_question, top_k=6)
_contexts = rerank(latency_question, _candidates, top_k=3)
t1 = time.perf_counter()
_ = generate_answer(latency_question, _contexts)
t2 = time.perf_counter()

pd.DataFrame([
    {"stage": "retrieval (hybrid + rerank)", "seconds": round(t1 - t0, 3)},
    {"stage": "generation", "seconds": round(t2 - t1, 3)},
    {"stage": "total", "seconds": round(t2 - t0, 3)},
])


,stage,seconds
0,retrieval (hybrid + rerank),0.039
1,generation,8.183
2,total,8.221


## 11. System Metrics Report

A short summary of the configuration used, useful for the assignment write-up and for comparing
settings later (chunk size, embedding model/dimension, vector store, and generation model).


In [ ]:
metrics_report = {
    "num_source_documents": len(raw_documents),
    "num_chunks": len(all_chunks),
    "chunk_size_words": CHUNK_SIZE,
    "chunk_overlap_words": CHUNK_OVERLAP,
    "embedding_model": "sentence-transformers/all-MiniLM-L6-v2",
    "embedding_dimension": EMBED_DIM,
    "vector_store": "FAISS (IndexFlatIP, cosine via inner product on normalized vectors)",
    "keyword_search": "BM25Okapi (rank_bm25)",
    "reranker_model": "cross-encoder/ms-marco-MiniLM-L-6-v2",
    "generation_model": "google/flan-t5-base",
    "embedding_model_alt_tested": "paraphrase-MiniLM-L3-v2",
    "generation_model_alt_tested": "google/flan-t5-small",
}

pd.Series(metrics_report).to_frame("value")


,value
num_source_documents,1
num_chunks,7
chunk_size_words,120
chunk_overlap_words,2
embedding_model,sentence-transformers/all-MiniLM-L6-v2
embedding_dimension,384
vector_store,"FAISS (IndexFlatIP, cosine via inner product on normalized vectors)"
keyword_search,BM25Okapi (rank_bm25)
reranker_model,cross-encoder/ms-marco-MiniLM-L-6-v2
generation_model,google/flan-t5-base


## Notes / Learnings

- **Chunk size trade-off:** smaller chunks retrieve more precisely but lose surrounding context;
  larger chunks keep context together but can dilute the similarity signal with irrelevant text.
  120 words per chunk, built up from whole sentences with a 2-sentence overlap, worked well for
  this notes-style document without ever cutting a sentence in half.
- **Why sentence-aware chunking mattered:** an earlier word-count-only version could start or end
  a chunk mid-sentence, which produced fragments that embedded and retrieved less coherently.
  Switching to whole-sentence chunks made retrieved passages noticeably more relevant.
- **Why hybrid search:** vector search alone occasionally missed chunks that shared exact keywords
  with the question but were phrased very differently; blending in BM25 fixed most of those misses.
- **Why re-rank:** the cross-encoder consistently pushed the single most relevant chunk to position
  1 even when the first-stage retrieval had it at position 2 or 3, which matters when only the top
  chunk or two are shown to the generator.
- **Generation gap observed:** retrieval is essentially perfect (hit_rate@3 = 1.0 across all
  methods), but end-to-end answer accuracy was only 20% (Section 10) — `flan-t5-base` sometimes
  echoes a raw context fragment (e.g. "[1]", a chapter heading) instead of synthesizing a full
  answer. The retrieval/grounding logic is working correctly (see the correct "don't know" in
  10.1); the bottleneck is generation quality. A larger instruction-tuned model such as
  `flan-t5-large` or a chat-tuned LLM would likely close this gap.
- **Embedding model comparison:** swapping `all-MiniLM-L6-v2` for `paraphrase-MiniLM-L3-v2` and
  comparing `hit_rate@3` on the same eval set showed both models tied at hit_rate@3 = 1.0 on this eval set — no clear winner here,
  so `paraphrase-MiniLM-L3-v2` is a valid lighter-weight swap for this corpus.
- **Language model comparison:** `flan-t5-base` vs `flan-t5-small` on identical retrieved context
  (Section 8.1) showed both models scored an identical 20% (1/5) answer accuracy on the eval set —
  the smaller model lost nothing in accuracy here, though this is a small sample.
- **Failure mode observed:** when a question asks about something not in the document at all, the
  prompt's explicit "say you don't know" instruction is what keeps the model from hallucinating an
  answer — removing that instruction produced more confident-sounding but ungrounded answers.
- **Swapping in your own documents:** replace the contents of `DOC_PATHS` in Section 1 with your own
  `.txt`/`.pdf` files (resume, class notes, a research paper, etc.) — no other cell needs to change.
